In [1]:
import numpy as np
from sympy import Matrix
from IPython.display import display

def output_matrix(X: np.ndarray, precision: int = 4):
    """
    Prints a NumPy array (vector or matrix) in LaTeX mathematical format using SymPy.
    
    Parameters:
    - X: np.ndarray, 1D or 2D array.
    - precision: number of decimal places to round floats to.
    """
    display(Matrix(np.round(X, precision)))

def read_matrix_from_file(filename, matrix_name="Ma trận"):
    """
    Read a matrix from a text file and display it.
    
    Parameters:
    filename (str): Path to the text file.
    matrix_name (str): The name of the matrix for display purposes.
    
    Returns:
    numpy.ndarray: The matrix read from the file
    """
    try:
        # Read the matrix from the file
        matrix = np.loadtxt(filename)
        
        # Display the matrix
        print(f"{matrix_name} đọc từ file:")
        output_matrix(matrix)
        
        return matrix
    
    except FileNotFoundError:
        print(f"Lỗi: File '{filename}' không được tìm thấy.")
        return None
    except Exception as e:
        print(f"Lỗi khi đọc file: {str(e)}")
        return None

# Phân tách Cholesky giải hệ tuyến tính


# Thuật toán
### 1. Phân tách Cholesky
**Input**: $A_{n \times n}$

**Output**: $U$

1. Kiểm tra nếu $A \neq A^T \to$ dừng chương trình, trái lại sang bước 2

2. Khởi tạo $U=[0]_{n \times n}$

3. Kiểm tra nếu $a_{11}>0$, gán $u_{11}=\sqrt{a_{11}}$, trái lại dừng chương trình

4. Cho $j$ chạy từ $2 \to n$, tính $u_{1j}=\dfrac{a_{1j}}{u_{11}}$

5. Cho $i$ chạy từ $2 \to n$, thực hiện:

- Gán temp = $a_{ii} - \displaystyle\sum\limits_{k=1}^{i-1}u^2_{ki}$, kiểm tra nếu temp $\le 0 \to$ dừng chương trình

- Gán $u_{ii} = \sqrt{\text{temp}}$

- Cho $j$ chạy từ $i+1 \to n$, gán: $u_{ij}=\dfrac{a_{ij}-\displaystyle\sum\limits_{k=1}^{i-1}u_{ki} \cdot u_{kj}}{u_{ii}}$

6. In ra $U$

### 2. Phân tách Cholesky giải hệ $AX=B$
**Input**: $A_{n \times n}$ và $B_{n \times m}$

**Output**: Nghiệm $X$ của hệ phương trình

1. Thực hiện tách Cholesky cho $A$. Nếu không tách được, nhân 2 vế của $AX=B$ với $A^T$ và tách Cholesky cho $M=A^T$, gán $B=A^T \cdot B$
2. Khởi tạo $Y=[0]_{n \times m}$
3. Cho $j$ chạy từ $1 \to m$, gán: $y_{ij}=\dfrac{b_{1j}}{u_{11}}$
4. Cho $i$ chạy từ $2 \to n$, $j$ chạy từ $1 \to m$, gán: $y_{ij} = \dfrac{b_{ij}-\displaystyle\sum\limits_{k=1}^{i-1}l_{ik} \cdot y_{kj}}{u_{ii}}$
5. Cho $j$ chạy từ $1 \to m$, gán: $x_{nj} = \dfrac{y_{nj}}{u_{nn}}$
6. Cho $i$ chạy từ $n-1 \to 1$, $j$ chạy từ $1 \to m$, gán: $x_{ij} = \dfrac{y_{ij}-\displaystyle\sum\limits_{k=i+1}^{n}u_{ik} \cdot x_{kj}}{u_{ii}}$
7. In ra nghiệm $X$

In [2]:
def cholesky_decomposition(A):
    """
    Phân tách Choleski ma trận A thành ma trận tam giác trên U (Câu 29).
    
    Parameters:
    A (numpy.ndarray): Ma trận đầu vào A
    
    Returns:
    numpy.ndarray: Ma trận tam giác trên U thỏa mãn A = U^T × U
    """
    n = len(A)
    
    # 2. Kiểm tra A = A^T
    if not np.allclose(A, A.T):
        raise ValueError("Không tồn tại cách tách Choleski (A không đối xứng)")
        
    # 3. Khởi tạo ma trận U
    U = np.zeros((n, n))
    
    # 4. Kiểm tra A[1][1] > 0
    if A[0][0] <= 0:
        raise ValueError("Không tồn tại cách tách Choleski (A[1][1] <= 0)")
        
    # 5. Gán U[1][1]
    U[0][0] = np.sqrt(A[0][0])
    
    # Bổ sung phần thuật toán bị thiếu trong PDF: Tính phần tử còn lại của hàng 1
    for j in range(1, n):
        U[0][j] = A[0][j] / U[0][0]
    
    print("U_1:")
    output_matrix(U[0, :].reshape(1, -1))
    
    # 6. Vòng lặp tổng quát
    for i in range(1, n):
        # 6.1: Gán tmp
        tmp = A[i][i] - sum(U[k][i]**2 for k in range(i))
        
        # 6.2: Kiểm tra tmp > 0
        if tmp <= 0:
            raise ValueError(f"Không tồn tại cách tách Choleski (tmp <= 0 tại i={i+1})")
            
        # 6.3: Gán U[i][i]
        U[i][i] = np.sqrt(tmp)
        
        # 6.4: Tính các phần tử còn lại của hàng i
        for j in range(i + 1, n):
            U[i][j] = (A[i][j] - sum(U[k][i] * U[k][j] for k in range(i))) / U[i][i]
            
        print(f"U_{i + 1}:")
        output_matrix(U[i, :].reshape(1, -1))
    
    return U


In [3]:
def verify_cholesky_decomposition(A, U):
    """
    Verify Cholesky decomposition by multiplying U^T and U and comparing with original matrix.
    
    Parameters:
    A (numpy.ndarray): Original matrix
    U (numpy.ndarray): Upper triangular matrix
    """
    print("\nUpper triangular matrix U:")
    output_matrix(U)
    print("\nLower triangular matrix U^T:")
    output_matrix(U.T)
    
    # Multiply U^T and U
    UT_U = np.dot(U.T, U)
    
    # Check if U^T × U equals A
    if np.allclose(UT_U, A):
        print("\nVerification successful: U^T × U = A")
    else:
        print("\nVerification failed: U^T × U ≠ A")

def cholesky_solve(A, B):
    """
    Giải hệ phương trình AX = B bằng phân tách Choleski (Câu 30).
    Nếu A không đối xứng, tự động chuyển về hệ M = A^T * A.
    
    Parameters:
    A (numpy.ndarray): Ma trận hệ số đầu vào
    B (numpy.ndarray): Ma trận vế phải
    
    Returns:
    numpy.ndarray: Ma trận nghiệm X
    """
    n = A.shape[0]
    cols = B.shape[1] if len(B.shape) > 1 else 1
    B_mat = B.copy().reshape(n, cols)
    
    print("\n--- BẮT ĐẦU THUẬT TOÁN GIẢI HỆ BẰNG CHOLESKI ---")
    # 3. Kiểm tra A = A^T
    if np.allclose(A, A.T):
        print("=> Ma trận A đối xứng, tiến hành phân rã trực tiếp A.")
        U = cholesky_decomposition(A)
        verify_cholesky_decomposition(A, U)
    else:
        print("=> Ma trận A không đối xứng, chuyển hệ về M = A^T * A và D = A^T * B.")
        M = np.dot(A.T, A)
        B_mat = np.dot(A.T, B_mat)
        print("\nMa trận M:")
        output_matrix(M)
        print("\nMa trận D (Vế phải mới):")
        output_matrix(B_mat)
        U = cholesky_decomposition(M)
        verify_cholesky_decomposition(M, U)
        
    print("\n--- GIẢI HỆ PHƯƠNG TRÌNH ---")
    print("Bước 1: Tính ma trận y (thế xuôi U^T * y = Vế phải)")
    
    y = np.zeros_like(B_mat, dtype=float)
    for j in range(cols):
        y[0, j] = B_mat[0, j] / U[0, 0]
    print("y_1:")
    output_matrix(y[0, :].reshape(1, -1))
        
    for i in range(1, n):
        for j in range(cols):
            sum_val = sum(U[k, i] * y[k, j] for k in range(i))
            y[i, j] = (B_mat[i, j] - sum_val) / U[i, i]
        print(f"y_{i + 1}:")
        output_matrix(y[i, :].reshape(1, -1))
        
    print("\nMa trận y hoàn chỉnh:")
    output_matrix(y)
            
    print("\nBước 2: Tính ma trận X (thế ngược UX = y)")
    X = np.zeros_like(y, dtype=float)
    for j in range(cols):
        X[n - 1, j] = y[n - 1, j] / U[n - 1, n - 1]
    print(f"X_{n}:")
    output_matrix(X[n - 1, :].reshape(1, -1))
        
    for i in range(n - 2, -1, -1):
        for j in range(cols):
            sum_val = sum(U[i, k] * X[k, j] for k in range(i + 1, n))
            X[i, j] = (y[i, j] - sum_val) / U[i, i]
        print(f"X_{i + 1}:")
        output_matrix(X[i, :].reshape(1, -1))
            
    return X

In [4]:
# Example usage
A = read_matrix_from_file('A34.txt', matrix_name="Ma trận A")
B = read_matrix_from_file('B34.txt', matrix_name="Ma trận B")

if A is not None and B is not None:
    if len(B.shape) == 1:
        B = B.reshape(-1, 1)
        
    try:
        # Giải hệ phương trình (bao gồm cả phân rã Choleski theo Câu 30)
        X = cholesky_solve(A, B)
        
        print("\n=> KẾT LUẬN NGHIỆM TỔNG QUÁT DƯỚI DẠNG MA TRẬN:")
        print("Hệ có nghiệm duy nhất X =")
        output_matrix(X)
    except ValueError as e:
        print(f"Lỗi: {e}")

Lỗi: File 'A34.txt' không được tìm thấy.
Lỗi: File 'B34.txt' không được tìm thấy.


In [5]:
# Test với ma trận C (không đối xứng) và D
C = read_matrix_from_file('CLSK_input_C.txt', matrix_name="Ma trận C")
D = read_matrix_from_file('CLSK_input_D.txt', matrix_name="Ma trận D")

if C is not None and D is not None:
    if len(D.shape) == 1:
        D = D.reshape(-1, 1)
        
    try:
        # Gọi cholesky_solve, hàm sẽ tự động nhận diện C không đối xứng và nhân C^T
        X_new = cholesky_solve(C, D)
        
        print("\n=> KẾT LUẬN NGHIỆM TỔNG QUÁT DƯỚI DẠNG MA TRẬN:")
        print("Hệ có nghiệm duy nhất X =")
        output_matrix(X_new)
    except ValueError as e:
        print(f"Lỗi: {e}")

Ma trận C đọc từ file:


Matrix([
[ 1.0, -2.0, 3.0],
[ 2.0,  4.0, 1.0],
[-1.0,  3.0, 5.0]])

Ma trận D đọc từ file:


Matrix([
[-6.0],
[ 9.0],
[ 0.0]])


--- BẮT ĐẦU THUẬT TOÁN GIẢI HỆ BẰNG CHOLESKI ---
=> Ma trận A không đối xứng, chuyển hệ về M = A^T * A và D = A^T * B.

Ma trận M:


...

Matrix([[2.0]])

X_1:


Matrix([[1.0]])


=> KẾT LUẬN NGHIỆM TỔNG QUÁT DƯỚI DẠNG MA TRẬN:
Hệ có nghiệm duy nhất X =


Matrix([
[ 1.0],
[ 2.0],
[-1.0]])